# Exploratory Data Analysis — ANRF AISEHack Theme 2
## PM2.5 Concentration Forecasting over India

**Objective:** Understand the dataset deeply before modelling — spatial structure, seasonal patterns, feature value ranges, correlations with cpm25, sparsity, and test data properties.

**Runs locally on CPU — no Kaggle GPU needed.**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

# ── Point this to your local data folder ──
DATA = os.path.abspath('../../../aisehack-theme-2')
# If running on Kaggle:
# DATA = '/kaggle/input/competitions/aisehack-theme-2'

MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
SEASON_LABELS = {'APRIL_16': 'Apr (Summer)', 'JULY_16': 'Jul (Monsoon)',
                 'OCT_16': 'Oct (Post-monsoon)', 'DEC_16': 'Dec (Winter)'}
MET_FEATS = ['u10', 'v10', 'pblh', 'rain', 't2', 'q2', 'swdown', 'psfc']
EMIS_FEATS = ['PM25', 'SO2', 'NOx', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio']
ALL_FEATS = ['cpm25'] + MET_FEATS + EMIS_FEATS

print(f'Data path: {DATA}')
print(f'Features ({len(ALL_FEATS)}): {ALL_FEATS}')

ModuleNotFoundError: No module named 'matplotlib'

---
## 1. Dataset Overview — Shapes, Dtypes & Time Ranges

In [ ]:
# ── Shapes and dtypes per feature per month ──
print(f'{"Feature":15s} {"Shape":22s} {"Dtype":10s}')
print('-' * 50)
for f in sorted(os.listdir(os.path.join(DATA, 'raw', 'APRIL_16'))):
    arr = np.load(os.path.join(DATA, 'raw', 'APRIL_16', f), allow_pickle=True)
    print(f'{f:15s} {str(arr.shape):22s} {str(arr.dtype):10s}')

print()
print('─── Time Ranges ───')
for m in MONTHS:
    t = np.load(os.path.join(DATA, 'raw', m, 'time.npy'), allow_pickle=True)
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy'))
    print(f'{m:12s}  {cpm.shape[0]:4d} hours  |  {t[0]}  →  {t[-1]}')

In [ ]:
# ── Lat/Lon grid ──
ll = np.load(os.path.join(DATA, 'raw', 'lat_long.npy'))
lat, lon = ll[:, :, 0], ll[:, :, 1]
print(f'Grid: {lat.shape[0]} x {lat.shape[1]}  |  25×25 km resolution')
print(f'Lat range: {lat.min():.2f}°N  →  {lat.max():.2f}°N')
print(f'Lon range: {lon.min():.2f}°E  →  {lon.max():.2f}°E')

---
## 2. Feature Statistics Summary

In [ ]:
# ── Comprehensive stats table ──
rows = []
for feat in ALL_FEATS:
    arrs = []
    for m in MONTHS:
        arrs.append(np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float64))
    v = np.concatenate(arrs, axis=0)
    nz_pct = np.count_nonzero(v) / v.size * 100
    rows.append({
        'Feature': feat,
        'Min': v.min(), 'Max': v.max(),
        'Mean': v.mean(), 'Std': v.std(),
        'Median': np.median(v),
        'P95': np.percentile(v, 95),
        'P99': np.percentile(v, 99),
        'Nonzero%': nz_pct,
    })
    del v

# Print formatted table
print(f'{"Feature":15s} {"Min":>12s} {"Max":>12s} {"Mean":>12s} {"Std":>12s} {"P95":>12s} {"P99":>12s} {"NZ%":>7s}')
print('─' * 100)
for r in rows:
    print(f'{r["Feature"]:15s} {r["Min"]:12.4e} {r["Max"]:12.4e} {r["Mean"]:12.4e} '
          f'{r["Std"]:12.4e} {r["P95"]:12.4e} {r["P99"]:12.4e} {r["Nonzero%"]:6.1f}%')

### Key Observations
- **cpm25** (target): right-skewed, mean ~34 µg/m³, but extreme events reach 2849 µg/m³
- **Emission features** are in extremely small magnitudes (~1e-8 to 1e-11) — need careful normalization
- **rain** is 70% zeros (sparse), **swdown** is 50% zeros (nighttime), **NMVOC_finn** is 98% zeros, **bio** is 64% zeros
- **psfc** has a huge absolute range (48k–102k Pa) but carries topography info

---
## 3. Spatial Patterns — cpm25 Heatmaps by Season

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, m in enumerate(MONTHS):
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    time = np.load(os.path.join(DATA, 'raw', m, 'time.npy'), allow_pickle=True)

    # Time-averaged spatial mean
    spatial_mean = cpm.mean(axis=0)
    im1 = axes[0, col].imshow(spatial_mean[::-1], cmap='YlOrRd',
                              norm=mcolors.LogNorm(vmin=1, vmax=300))
    axes[0, col].set_title(f'{SEASON_LABELS[m]}\nTemporal Mean', fontsize=11)
    axes[0, col].axis('off')
    plt.colorbar(im1, ax=axes[0, col], fraction=0.046, label='µg/m³')

    # Snapshot at a high-pollution hour (pick 75th percentile hour)
    hourly_means = cpm.mean(axis=(1, 2))
    peak_idx = np.argsort(hourly_means)[int(0.95 * len(hourly_means))]
    im2 = axes[1, col].imshow(cpm[peak_idx, ::-1], cmap='YlOrRd',
                              norm=mcolors.LogNorm(vmin=1, vmax=500))
    axes[1, col].set_title(f'P95 Peak Hour\n{time[peak_idx]}', fontsize=10)
    axes[1, col].axis('off')
    plt.colorbar(im2, ax=axes[1, col], fraction=0.046, label='µg/m³')

fig.suptitle('PM2.5 Spatial Patterns by Season', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_spatial_cpm25.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Seasonal cpm25 Distribution (Histograms + Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Histograms ──
for m in MONTHS:
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    # Subsample for speed
    flat = cpm.reshape(-1)
    sample = flat[np.random.choice(len(flat), 500_000, replace=False)]
    axes[0].hist(sample, bins=200, alpha=0.5, label=SEASON_LABELS[m],
                 range=(0, 300), density=True)

axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].set_ylabel('Density')
axes[0].set_title('cpm25 Distribution by Season')
axes[0].legend()
axes[0].set_xlim(0, 300)

# ── Box plots ──
box_data = []
box_labels = []
for m in MONTHS:
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    flat = cpm.reshape(-1)
    sample = flat[np.random.choice(len(flat), 100_000, replace=False)]
    box_data.append(sample)
    box_labels.append(SEASON_LABELS[m])

axes[1].boxplot(box_data, labels=box_labels, showfliers=False)
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].set_title('cpm25 Box Plots (no outliers)')

plt.tight_layout()
plt.savefig('../outputs/eda_cpm25_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Per-Season Summary ──')
for m in MONTHS:
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    print(f'{SEASON_LABELS[m]:25s}  mean={cpm.mean():.1f}  std={cpm.std():.1f}  '
          f'p50={np.median(cpm):.1f}  p95={np.percentile(cpm,95):.1f}  '
          f'p99={np.percentile(cpm,99):.1f}  max={cpm.max():.1f}')

**Insight:** December (Winter) has 2–3× higher mean PM2.5 than July (Monsoon), with much heavier tails. This is the seasonal variability the model must capture.

---
## 5. Temporal Trends — Domain-Average cpm25 Over Time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 8), sharey=True)
axes = axes.ravel()

for idx, m in enumerate(MONTHS):
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    domain_mean = cpm.mean(axis=(1, 2))      # (T,)
    domain_max  = cpm.max(axis=(1, 2))
    hours = np.arange(len(domain_mean))

    axes[idx].fill_between(hours, 0, domain_mean, alpha=0.3, color='steelblue')
    axes[idx].plot(hours, domain_mean, 'steelblue', linewidth=0.8, label='Domain Mean')
    axes[idx].plot(hours, domain_max, 'red', linewidth=0.5, alpha=0.5, label='Domain Max')
    axes[idx].set_title(SEASON_LABELS[m])
    axes[idx].set_xlabel('Hour')
    axes[idx].legend(fontsize=8)
    axes[idx].set_ylim(0, 400)

fig.suptitle('Domain-Average PM2.5 Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_temporal_cpm25.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Diurnal Cycle — cpm25 Hour-of-Day Patterns

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for idx, m in enumerate(MONTHS):
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    time = np.load(os.path.join(DATA, 'raw', m, 'time.npy'), allow_pickle=True)

    # Extract hour of day from timestamp strings
    hours_of_day = np.array([int(str(t).split('T')[1].split(':')[0]) for t in time])

    domain_mean = cpm.mean(axis=(1, 2))

    # Group by hour
    hourly_avgs = []
    hourly_stds = []
    for h in range(24):
        mask = hours_of_day == h
        if mask.sum() > 0:
            hourly_avgs.append(domain_mean[mask].mean())
            hourly_stds.append(domain_mean[mask].std())
        else:
            hourly_avgs.append(0)
            hourly_stds.append(0)

    hourly_avgs = np.array(hourly_avgs)
    hourly_stds = np.array(hourly_stds)

    axes[idx].bar(range(24), hourly_avgs, alpha=0.7, color='coral')
    axes[idx].errorbar(range(24), hourly_avgs, yerr=hourly_stds,
                       fmt='none', color='black', capsize=2, linewidth=0.5)
    axes[idx].set_title(SEASON_LABELS[m], fontsize=10)
    axes[idx].set_xlabel('Hour (UTC)')
    if idx == 0:
        axes[idx].set_ylabel('Domain Mean PM2.5')
    axes[idx].set_xticks(range(0, 24, 3))

fig.suptitle('Diurnal Cycle of PM2.5 (UTC)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_diurnal_cpm25.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** Expect a diurnal peak matching India's nighttime (low boundary layer). The model benefits from capturing this cycle through the met variable `pblh`.

---
## 7. Spatial Maps of All Meteorological Features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

m = 'DEC_16'  # Winter — most interesting for PM2.5
for idx, feat in enumerate(MET_FEATS):
    arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float32)
    spatial_mean = arr.mean(axis=0)
    ax = axes[idx // 4, idx % 4]
    im = ax.imshow(spatial_mean[::-1], cmap='viridis')
    ax.set_title(f'{feat}', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.suptitle(f'Time-Averaged Meteorological Fields — {SEASON_LABELS[m]}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_met_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Feature Correlation with cpm25 (Spatial Average)

In [ ]:
# Compute temporal correlation between domain-averaged features and cpm25
# Using DEC_16 (highest PM2.5 variability)

m = 'DEC_16'
cpm_ts = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32).mean(axis=(1, 2))

corr_results = {}
for feat in MET_FEATS + EMIS_FEATS:
    arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float64)
    feat_ts = arr.mean(axis=(1, 2))
    corr = np.corrcoef(cpm_ts, feat_ts)[0, 1]
    corr_results[feat] = corr

# Sort by absolute correlation
sorted_corr = sorted(corr_results.items(), key=lambda x: abs(x[1]), reverse=True)

fig, ax = plt.subplots(figsize=(10, 6))
feats_sorted = [x[0] for x in sorted_corr]
corrs_sorted = [x[1] for x in sorted_corr]
colors = ['#d62728' if c < 0 else '#2ca02c' for c in corrs_sorted]

bars = ax.barh(feats_sorted[::-1], corrs_sorted[::-1], color=colors[::-1])
ax.set_xlabel('Pearson Correlation with cpm25')
ax.set_title(f'Feature Correlation with PM2.5 — {SEASON_LABELS[m]}')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlim(-1, 1)

for i, (feat, corr) in enumerate(zip(feats_sorted[::-1], corrs_sorted[::-1])):
    ax.text(corr + 0.02 * np.sign(corr), i, f'{corr:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n── Correlation Ranking ──')
for feat, corr in sorted_corr:
    bar = '█' * int(abs(corr) * 40)
    sign = '+' if corr > 0 else '-'
    print(f'{feat:15s} {sign}{abs(corr):.3f}  {bar}')

---
## 9. Pixel-Level Correlation Map (cpm25 vs Key Features)

In [ ]:
# Per-pixel temporal correlation between cpm25 and top met features
m = 'DEC_16'
cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
top_feats = ['t2', 'pblh', 'q2', 'rain']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, feat in enumerate(top_feats):
    arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float32)

    # Pixel-wise Pearson correlation across time
    T, H, W = cpm.shape
    cpm_flat = cpm.reshape(T, -1)  # (T, H*W)
    arr_flat = arr.reshape(T, -1)

    # Standardize
    cpm_z = (cpm_flat - cpm_flat.mean(0)) / (cpm_flat.std(0) + 1e-8)
    arr_z = (arr_flat - arr_flat.mean(0)) / (arr_flat.std(0) + 1e-8)

    corr_map = (cpm_z * arr_z).mean(0).reshape(H, W)

    im = axes[idx].imshow(corr_map[::-1], cmap='RdBu_r', vmin=-1, vmax=1)
    axes[idx].set_title(f'corr(cpm25, {feat})', fontsize=11)
    axes[idx].axis('off')
    plt.colorbar(im, ax=axes[idx], fraction=0.046)

fig.suptitle(f'Pixel-Level Temporal Correlation — {SEASON_LABELS[m]}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_pixel_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Emission Feature Analysis (Very Small Values)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

m = 'DEC_16'
for idx, feat in enumerate(EMIS_FEATS):
    arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float64)
    spatial_mean = arr.mean(axis=0)
    nz_pct = (spatial_mean > 0).sum() / spatial_mean.size * 100

    ax = axes[idx // 4, idx % 4]
    if spatial_mean.max() > 0:
        im = ax.imshow(spatial_mean[::-1], cmap='hot',
                       norm=mcolors.LogNorm(vmin=max(spatial_mean[spatial_mean > 0].min(), 1e-15),
                                            vmax=spatial_mean.max()))
    else:
        im = ax.imshow(spatial_mean[::-1], cmap='hot')
    ax.set_title(f'{feat}\nmax={spatial_mean.max():.2e} | {nz_pct:.0f}% nonzero', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Hide unused subplot
axes[1, 3].axis('off')

fig.suptitle(f'Emission Features Spatial Average — {SEASON_LABELS[m]}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_emissions.png', dpi=150, bbox_inches='tight')
plt.show()

# Value range comparison
print('\n── Emission Feature Magnitudes (all months combined) ──')
for feat in EMIS_FEATS:
    arrs = []
    for mo in MONTHS:
        arrs.append(np.load(os.path.join(DATA, 'raw', mo, f'{feat}.npy')).astype(np.float64))
    v = np.concatenate(arrs, axis=0)
    nz = np.count_nonzero(v) / v.size * 100
    print(f'{feat:15s}  max={v.max():.2e}  mean={v.mean():.2e}  nonzero={nz:.1f}%')

**Key finding:** Emission values are O(1e-8 to 1e-11). They're almost constant across time (proxy emission inventories, not dynamic). `NMVOC_finn` (biomass burning) is 98% zeros — highly sparse. Grid-wise normalization is essential for these.

---
## 11. Feature Sparsity Heatmap

In [ ]:
# Sparsity (% zeros) per feature per month
sparsity = np.zeros((len(ALL_FEATS), len(MONTHS)))

for i, feat in enumerate(ALL_FEATS):
    for j, m in enumerate(MONTHS):
        arr = np.load(os.path.join(DATA, 'raw', m, f'{feat}.npy')).astype(np.float32)
        sparsity[i, j] = (arr == 0).sum() / arr.size * 100

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(sparsity, annot=True, fmt='.0f', cmap='YlOrRd',
            xticklabels=[SEASON_LABELS[m] for m in MONTHS],
            yticklabels=ALL_FEATS, ax=ax,
            cbar_kws={'label': '% Zero Values'})
ax.set_title('Feature Sparsity by Season (% Zeros)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Wind Field Visualization

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, m in enumerate(MONTHS):
    u = np.load(os.path.join(DATA, 'raw', m, 'u10.npy')).astype(np.float32)
    v = np.load(os.path.join(DATA, 'raw', m, 'v10.npy')).astype(np.float32)

    # Time-averaged wind speed and direction
    u_mean = u.mean(axis=0)
    v_mean = v.mean(axis=0)
    wspd = np.sqrt(u_mean**2 + v_mean**2)

    ax = axes[idx]
    im = ax.imshow(wspd[::-1], cmap='coolwarm', vmin=0, vmax=8)
    # Quiver plot (subsample for readability)
    step = 8
    Y, X = np.mgrid[0:140:step, 0:124:step]
    ax.quiver(X, 140 - Y - 1, u_mean[::step, ::step], v_mean[::step, ::step],
              color='black', scale=80, width=0.003, alpha=0.7)
    ax.set_title(SEASON_LABELS[m], fontsize=11)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, label='m/s')

fig.suptitle('Mean Wind Speed & Direction by Season', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_winds.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. PBL Height vs PM2.5 — The Key Physical Relationship

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, m in enumerate(MONTHS):
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    pblh = np.load(os.path.join(DATA, 'raw', m, 'pblh.npy')).astype(np.float32)

    # Domain-averaged time series
    cpm_ts = cpm.mean(axis=(1, 2))
    pbl_ts = pblh.mean(axis=(1, 2))

    ax = axes[idx]
    ax2 = ax.twinx()
    ax.plot(cpm_ts[:72], 'coral', linewidth=1, label='cpm25')
    ax2.plot(pbl_ts[:72], 'steelblue', linewidth=1, label='PBLH')
    ax.set_ylabel('PM2.5', color='coral')
    ax2.set_ylabel('PBLH (m)', color='steelblue')
    ax.set_title(SEASON_LABELS[m], fontsize=10)
    ax.set_xlabel('Hour')

    # Add correlation value
    corr = np.corrcoef(cpm_ts, pbl_ts)[0, 1]
    ax.text(0.05, 0.95, f'r={corr:.2f}', transform=ax.transAxes,
            fontsize=9, va='top', bbox=dict(boxstyle='round', fc='white', alpha=0.8))

fig.suptitle('PM2.5 vs Boundary Layer Height (first 72h)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eda_pblh_vs_cpm25.png', dpi=150, bbox_inches='tight')
plt.show()

**Physics:** When PBLH is low (nighttime), pollutants are trapped near the surface → high PM2.5. When PBLH rises (daytime heating), mixing dilutes PM2.5. Expect strong negative correlation.

---
## 14. Autocorrelation of cpm25 (Temporal Predictability)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

max_lag = 48  # hours
for idx, m in enumerate(MONTHS):
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    ts = cpm.mean(axis=(1, 2))

    autocorrs = []
    for lag in range(max_lag + 1):
        if lag == 0:
            autocorrs.append(1.0)
        else:
            autocorrs.append(np.corrcoef(ts[:-lag], ts[lag:])[0, 1])

    axes[idx].bar(range(max_lag + 1), autocorrs, color='steelblue', alpha=0.7)
    axes[idx].axhline(0, color='black', linewidth=0.5)
    axes[idx].axvline(10, color='red', linestyle='--', alpha=0.7, label='t=10 (last cpm25 input)')
    axes[idx].axvline(26, color='green', linestyle='--', alpha=0.7, label='t=26 (forecast starts)')
    axes[idx].set_title(SEASON_LABELS[m], fontsize=10)
    axes[idx].set_xlabel('Lag (hours)')
    axes[idx].set_ylim(-0.2, 1.05)
    if idx == 0:
        axes[idx].set_ylabel('Autocorrelation')
        axes[idx].legend(fontsize=7)

fig.suptitle('cpm25 Autocorrelation (Domain Mean)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_autocorrelation.png', dpi=150, bbox_inches='tight')
plt.show()

**Insight:** High autocorrelation at short lags → recent cpm25 history is predictive. The red line shows where cpm25 input ends (t=10); the model must rely on met features after that.

---
## 15. Test Data Analysis

In [ ]:
print('=== Test Input Shapes ===')
for f in sorted(os.listdir(os.path.join(DATA, 'test_in'))):
    arr = np.load(os.path.join(DATA, 'test_in', f))
    print(f'{f:20s} {str(arr.shape):25s} dtype={arr.dtype}')

# cpm25 test — compare distribution with training
test_cpm = np.load(os.path.join(DATA, 'test_in', 'cpm25.npy'))  # (996, 10, 140, 124)
print(f'\nTest cpm25: {test_cpm.shape}')
print(f'  min={test_cpm.min():.4f}  max={test_cpm.max():.4f}  mean={test_cpm.mean():.4f}')

In [ ]:
# Compare test vs train distributions of cpm25
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Test distribution ──
test_flat = test_cpm.reshape(-1)
sample_test = test_flat[np.random.choice(len(test_flat), 500_000, replace=False)]

axes[0].hist(sample_test, bins=200, range=(0, 300), alpha=0.6,
             density=True, color='steelblue', label='Test (2017)')

# Overlay each training month
for m in MONTHS:
    cpm = np.load(os.path.join(DATA, 'raw', m, 'cpm25.npy')).astype(np.float32)
    flat = cpm.reshape(-1)
    sample = flat[np.random.choice(len(flat), 200_000, replace=False)]
    axes[0].hist(sample, bins=200, range=(0, 300), alpha=0.3,
                 density=True, label=SEASON_LABELS[m])

axes[0].set_xlabel('PM2.5 (µg/m³)')
axes[0].set_ylabel('Density')
axes[0].set_title('Test vs Train cpm25 Distribution')
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, 300)

# ── Test sample spatial snapshots ──
# Show 4 random test samples at t=0
fig2, axes2 = plt.subplots(2, 4, figsize=(16, 8))
rng = np.random.default_rng(42)
sample_ids = rng.choice(996, 8, replace=False)

for i, sid in enumerate(sample_ids):
    ax = axes2[i // 4, i % 4]
    im = ax.imshow(test_cpm[sid, 0, ::-1], cmap='YlOrRd',
                   norm=mcolors.LogNorm(vmin=1, vmax=500))
    ax.set_title(f'Sample {sid} (t=0)', fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

fig2.suptitle('Random Test Input Snapshots (cpm25 at t=0)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_test_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 16. Test cpm25 — Temporal Evolution Within Input Window

In [ ]:
# How does domain-mean cpm25 evolve over the 10 input hours?
test_domain_mean = test_cpm.mean(axis=(2, 3))  # (996, 10)

fig, ax = plt.subplots(figsize=(10, 5))
# Plot percentiles across samples
hours = np.arange(10)
p25 = np.percentile(test_domain_mean, 25, axis=0)
p50 = np.percentile(test_domain_mean, 50, axis=0)
p75 = np.percentile(test_domain_mean, 75, axis=0)
p95 = np.percentile(test_domain_mean, 95, axis=0)
mean_ts = test_domain_mean.mean(axis=0)

ax.fill_between(hours, p25, p75, alpha=0.3, color='steelblue', label='IQR')
ax.plot(hours, p50, 'steelblue', linewidth=2, label='Median')
ax.plot(hours, mean_ts, 'coral', linewidth=2, linestyle='--', label='Mean')
ax.plot(hours, p95, 'red', linewidth=1, alpha=0.7, label='P95')
ax.set_xlabel('Input Hour (0–9)')
ax.set_ylabel('Domain Mean PM2.5 (µg/m³)')
ax.set_title('Test cpm25 Input Window — Temporal Spread Across 996 Samples')
ax.legend()
ax.set_xticks(hours)

plt.tight_layout()
plt.savefig('../outputs/eda_test_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Test samples: {test_cpm.shape[0]}')
print(f'Mean across all samples & hours: {test_cpm.mean():.2f} µg/m³')
print(f'This is closest to the Oct + Dec range in training data.')

---
## 17. Summary & Feature Selection Recommendations

### Dataset Summary

| Property | Value |
|----------|-------|
| Grid | 140 × 124 (~25km resolution) |
| Lat | 7.06°N – 38.00°N |
| Lon | 67.79°E – 97.93°E |
| Training months | Apr, Jul, Oct, Dec 2016 (~715–739 hours each) |
| Test | 996 samples from 2017, 10h cpm25 + 26h other features |
| Target | cpm25 next 16 hours → shape (996, 140, 124, 16) |

### Feature Priority for First Run

| Tier | Features | Rationale |
|------|----------|-----------|
| **Must-use** | cpm25, t2, pblh, u10, v10 | Strong physical drivers: temperature inversion, mixing height, wind transport |
| **High-value** | q2, rain | Humidity effects + wet deposition |
| **Include** | PM25, SO2, NOx | Emission sources — correlated with urban areas |
| **Low priority** | swdown, psfc | swdown is 50% zero (night); psfc is mostly topography (static proxy) |
| **Skip first** | NH3, NMVOC_e, NMVOC_finn, bio | Very sparse, tiny magnitudes, unlikely to help in 10-feature fast run |

### Key Physics to Encode
- PBLH and cpm25 are **anti-correlated** — low mixing heights trap pollution
- **Diurnal cycle** is strong — model needs to capture hourly periodicity
- **December** is the hardest month — highest PM2.5, most extreme events
- Test data appears to come from months with **moderate-to-high** pollution (Oct–Dec range)
- **Wind fields** provide advection signal — transport of pollution between grid cells

### Normalization Warning
- Emission features are O(1e-8 to 1e-11) — need **grid-wise z-score** normalization
- `rain` and `NMVOC_finn` are >70% zeros — consider log-transform or leaving them for later experiments